# 2. Getting the data

**What a TESS sector is.** TESS stares at one strip of sky for about 27 days (two 13.7-day orbits with a short download gap in the middle), recording total brightness every 30 minutes early in the mission, later every 10 minutes and then every 200 seconds. That 27-day stare is one **sector**. TESS then moves on and only comes back to the same strip months or years later. A star in our sample has been seen in 5 to 20 sectors.

**What TARS gives us.** The TARS team turned the raw TESS images into one brightness curve per star per sector, with the instrument's slow trends already removed, and published them as a High Level Science Product (HLSP) on NASA's MAST archive. One file per star per sector. We download those.

**How.** `astroquery` talks to MAST. We ask for every TARS product for a given TIC number and download the `_lc.fits` files. The cell below is the whole download recipe, demonstrated on one star; the following cell checks that every star in both samples has its files in `data/lightcurves/` and downloads any that are missing.

In [1]:
import sys; sys.path.insert(0, "..")
import json, time, os
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
import helpers as h
plt.rcParams["figure.dpi"] = 110
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)

In [2]:
from astroquery.mast import Observations

def download_star(tic_id, folder):
    """Download every TARS light-curve file for one star into folder. Returns the file paths."""
    folder = Path(folder); folder.mkdir(parents=True, exist_ok=True)
    observations = Observations.query_criteria(provenance_name="TARS", target_name=str(tic_id))
    if len(observations) == 0:
        return []
    products = Observations.get_product_list(observations)
    products = products[[str(f).endswith("_lc.fits") for f in products["productFilename"]]]
    manifest = Observations.download_products(products, download_dir=str(folder), cache=True)
    paths = []
    for local in manifest["Local Path"]:
        src = Path(local); dst = folder / src.name
        if src.exists() and not dst.exists():
            src.replace(dst)
        paths.append(dst)
    return sorted(paths)

# Demonstration on one star (the Sun-like example used throughout the write-up).
t0 = time.time()
try:
    demo = download_star(45468124, "../data/lightcurves/demo")
    print(f"{len(demo)} files for TIC 45468124 in {time.time()-t0:.0f} s; first one:", demo[0].name if demo else None)
except Exception as exc:
    print("Network unavailable right now:", type(exc).__name__, "- the cached files below are used instead.")

In [ ]:
def check_cache(sample_name, cache):
    """How many stars have every accepted sector on disk? Returns the list of stars with files missing."""
    missing = []
    for row in h.read_star_list(sample_name):
        on_disk = {int(p.name.split("_s")[1][:4]) for p in h.sector_files(cache, row["tic_id"])}
        if not row["sectors_allowed"] <= on_disk:
            missing.append((row["tic_id"], sorted(row["sectors_allowed"] - on_disk)))
    return missing

for sample_name, cache in [("original_450", h.DATA / "lightcurves" / "original"), ("replication_422", h.DATA / "lightcurves" / "replication")]:
    missing = check_cache(sample_name, cache)
    n_files = len(list(cache.glob("*_lc.fits")))
    print(f"{sample_name}: {n_files} sector files on disk; {len(missing)} stars with sectors missing")
    for tic_id, sectors in missing[:50]:          # download whatever is missing (nothing, if the cache is complete)
        try:
            download_star(tic_id, cache)
        except Exception as exc:
            print("  could not download", tic_id, type(exc).__name__)

In [ ]:
# What one sector looks like: the Sun-like example star, sector 9.
path = next(p for p in h.sector_files(h.DATA / "lightcurves" / "original", 45468124) if "_s0009" in p.name)
sector, t, f, noise = h.load_sector(path)
fig, ax = plt.subplots(figsize=(11, 3.4))
ax.plot(t - t.min(), 100 * (f - np.median(f)), ".", ms=2, color="#1f3b73")
ax.set_xlabel("days since the start of the sector"); ax.set_ylabel("brightness change (%)")
ax.set_title(f"TIC 45468124, TESS sector {sector}: {len(t)} points every {np.median(np.diff(t))*24*60:.0f} minutes; each dip is the spotted side facing us", fontsize=10)
ax.grid(alpha=0.2); fig.tight_layout(); fig.savefig("../figures/02_one_sector.png", dpi=150)
print(f"point-to-point noise recorded in the file: {100*noise:.2f}%")

**What to notice.** The wave repeats every ~2.9 days — that is the spot pattern turning — and its height visibly changes over the month. Nothing in this file tells us what the star's surface looks like; everything we know about its spots comes from this wave.